# Carried-phase checkpoint visualisation

Load a trained carried-phase checkpoint, generate synthetic spliced DNA, run the model, and plot true vs predicted phase in genomic register.

The model output classes are:

- `exon_phase_0`, `exon_phase_1`, `exon_phase_2`
- `intron_carry_0`, `intron_carry_1`, `intron_carry_2`

For downstream use, the three intron-carry classes collapse back to one `not_coding` track.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT_PATH = ROOT / "scripts" / "train_synthetic_splice_hybrid_phase_track.py"
CHECKPOINT_PATH = ROOT / "checkpoints" / "synthetic_splice_phase_track_hybrid_carried_bidir_l6_attn2" / "best.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo", ROOT)
print("checkpoint", CHECKPOINT_PATH)
print("device", DEVICE)

In [ ]:
spec = importlib.util.spec_from_file_location("phase_track", SCRIPT_PATH)
phase = importlib.util.module_from_spec(spec)
sys.modules["phase_track"] = phase
spec.loader.exec_module(phase)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
ckpt_args = checkpoint.get("args", {})
print("checkpoint step", checkpoint.get("step"))
print("checkpoint metrics", checkpoint.get("final_metrics"))
print("checkpoint labels", phase.PHASE_LABELS)

In [ ]:
model = phase.MambaPhaseTrackPredictor(
    hidden_dim=ckpt_args.get("hidden_dim", 64),
    layers=ckpt_args.get("layers", 3),
    chunk_size=ckpt_args.get("chunk_size", 32),
    headdim=ckpt_args.get("headdim", 8),
    local_conv_kernel=ckpt_args.get("local_conv_kernel", 9),
    head_conv_kernel=ckpt_args.get("head_conv_kernel", 7),
    bidirectional=ckpt_args.get("bidirectional", False),
    attention_layers=ckpt_args.get("attention_layers", 2),
    attention_heads=ckpt_args.get("attention_heads", 4),
    attention_window=ckpt_args.get("attention_window", 129),
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"parameters: {param_count:,}")

## Generate a synthetic test batch

Change `TEST_SEED`, `BATCH_SIZE`, or the exon/intron settings here to probe harder examples.

In [ ]:
TEST_SEED = 20260706
BATCH_SIZE = 4

batch = phase.make_batch(
    batch_size=BATCH_SIZE,
    device=torch.device(DEVICE),
    min_protein_codons=ckpt_args.get("min_protein_codons", 24),
    max_protein_codons=ckpt_args.get("max_protein_codons", 96),
    min_exon_count=ckpt_args.get("min_exon_count", 1),
    max_exon_count=ckpt_args.get("max_exon_count", 21),
    min_exon_bases=ckpt_args.get("min_exon_bases", 3),
    max_exon_bases=ckpt_args.get("max_exon_bases", 300),
    median_exon_bases=ckpt_args.get("median_exon_bases", 50),
    rare_short_exon_prob=ckpt_args.get("rare_short_exon_prob", 0.05),
    exon_length_mode=ckpt_args.get("exon_length_mode", "sampled"),
    min_intron_length=ckpt_args.get("min_intron_length", 50),
    max_intron_length=ckpt_args.get("max_intron_length", 300),
    length_bucket_size=ckpt_args.get("length_bucket_size", 4096),
    seed=TEST_SEED,
)

dna, splice_tracks, *_unused, examples = batch
targets, genome_mask = phase.phase_targets_from_examples(
    examples,
    max_length=dna.shape[1],
    device=torch.device(DEVICE),
)

with torch.no_grad():
    logits, encoded = model(dna, splice_tracks)
    probs = logits.softmax(dim=-1)
    pred = logits.argmax(dim=-1)

metrics = phase.phase_metrics(logits, targets, genome_mask)
metrics

In [ ]:
for i, example in enumerate(examples):
    genome_len = len(example["genome"])
    wrong_6 = ((pred[i] != targets[i]) & genome_mask[i]).sum().item()
    collapsed_pred = torch.where(pred[i] < 3, pred[i], torch.full_like(pred[i], 3))
    collapsed_target = torch.where(targets[i] < 3, targets[i], torch.full_like(targets[i], 3))
    wrong_4 = ((collapsed_pred != collapsed_target) & genome_mask[i]).sum().item()
    print(
        f"example {i}: genome={genome_len} bp, protein={example['protein_codons']} aa, "
        f"exons={example['exon_count']}, exon_lengths={example['exon_lengths']}, "
        f"6-state wrong={wrong_6}, collapsed wrong={wrong_4}"
    )

## Plot true vs predicted phase

Use `start` and `end` to zoom into an exon junction or an error region.

In [ ]:
STATE_COLORS = [
    "#d73027",  # exon phase 0
    "#fc8d59",  # exon phase 1
    "#fee08b",  # exon phase 2
    "#4575b4",  # intron carry 0
    "#91bfdb",  # intron carry 1
    "#e0f3f8",  # intron carry 2
]
STATE_CMAP = ListedColormap(STATE_COLORS)
STATE_NORM = BoundaryNorm(np.arange(-0.5, 6.5, 1), STATE_CMAP.N)

COLLAPSED_COLORS = ["#d73027", "#fc8d59", "#fee08b", "#d9d9d9"]
COLLAPSED_CMAP = ListedColormap(COLLAPSED_COLORS)
COLLAPSED_NORM = BoundaryNorm(np.arange(-0.5, 4.5, 1), COLLAPSED_CMAP.N)

def collapse_to_4(x):
    return torch.where(x < 3, x, torch.full_like(x, 3))

def first_error_window(example_index=0, pad=80, collapsed=False):
    real = genome_mask[example_index]
    if collapsed:
        bad = (collapse_to_4(pred[example_index]) != collapse_to_4(targets[example_index])) & real
    else:
        bad = (pred[example_index] != targets[example_index]) & real
    positions = torch.nonzero(bad, as_tuple=False).flatten().cpu().numpy()
    if len(positions) == 0:
        return 0, min(len(examples[example_index]["genome"]), 600)
    centre = int(positions[0])
    return max(0, centre - pad), min(len(examples[example_index]["genome"]), centre + pad)

def plot_phase(example_index=0, start=0, end=None):
    example = examples[example_index]
    genome_len = len(example["genome"])
    if end is None:
        end = genome_len
    start = max(0, int(start))
    end = min(genome_len, int(end))
    sl = slice(start, end)
    x = np.arange(start, end)

    true_6 = targets[example_index, sl].detach().cpu().numpy()
    pred_6 = pred[example_index, sl].detach().cpu().numpy()
    true_4 = collapse_to_4(targets[example_index, sl]).detach().cpu().numpy()
    pred_4 = collapse_to_4(pred[example_index, sl]).detach().cpu().numpy()
    confidence = probs[example_index, sl].max(dim=-1).values.detach().cpu().numpy()
    donor = splice_tracks[example_index, sl, 0].detach().cpu().numpy()
    acceptor = splice_tracks[example_index, sl, 1].detach().cpu().numpy()
    correct_6 = (true_6 == pred_6).astype(float)

    fig, axes = plt.subplots(
        6,
        1,
        figsize=(18, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [0.45, 0.45, 0.45, 0.45, 1.0, 1.0]},
    )

    axes[0].imshow(true_6[None, :], aspect="auto", cmap=STATE_CMAP, norm=STATE_NORM, extent=[start, end, 0, 1])
    axes[0].set_ylabel("true\n6-state", rotation=0, ha="right", va="center")
    axes[1].imshow(pred_6[None, :], aspect="auto", cmap=STATE_CMAP, norm=STATE_NORM, extent=[start, end, 0, 1])
    axes[1].set_ylabel("pred\n6-state", rotation=0, ha="right", va="center")
    axes[2].imshow(true_4[None, :], aspect="auto", cmap=COLLAPSED_CMAP, norm=COLLAPSED_NORM, extent=[start, end, 0, 1])
    axes[2].set_ylabel("true\n4-track", rotation=0, ha="right", va="center")
    axes[3].imshow(pred_4[None, :], aspect="auto", cmap=COLLAPSED_CMAP, norm=COLLAPSED_NORM, extent=[start, end, 0, 1])
    axes[3].set_ylabel("pred\n4-track", rotation=0, ha="right", va="center")

    axes[4].fill_between(x, 0, correct_6, step="pre", color="#1a9850", alpha=0.75, label="6-state correct")
    axes[4].fill_between(x, correct_6, 1, step="pre", color="#d73027", alpha=0.75, label="6-state wrong")
    axes[4].plot(x, confidence, color="black", lw=1.0, label="confidence")
    axes[4].set_ylim(-0.05, 1.05)
    axes[4].set_ylabel("correct\nconf", rotation=0, ha="right", va="center")
    axes[4].legend(loc="upper right", ncols=3, fontsize=8)

    axes[5].plot(x, donor, color="#762a83", lw=1.5, label="donor track")
    axes[5].plot(x, acceptor, color="#1b7837", lw=1.5, label="acceptor track")
    axes[5].set_ylim(-0.05, 1.05)
    axes[5].set_ylabel("splice\ntracks", rotation=0, ha="right", va="center")
    axes[5].legend(loc="upper right", fontsize=8)

    for ax in axes[:4]:
        ax.set_yticks([])
    axes[-1].set_xlabel("genomic position in synthetic example")
    fig.suptitle(
        f"Example {example_index}: genome={genome_len} bp, exons={example['exon_count']}, "
        f"protein={example['protein_codons']} aa, window={start}:{end}",
        y=1.02,
    )

    state_handles = [Patch(color=c, label=l) for c, l in zip(STATE_COLORS, phase.PHASE_LABELS)]
    collapsed_handles = [Patch(color=c, label=l) for c, l in zip(COLLAPSED_COLORS, ["phase0", "phase1", "phase2", "not coding"])]
    fig.legend(handles=state_handles + collapsed_handles, loc="lower center", ncols=5, bbox_to_anchor=(0.5, -0.08), fontsize=8)
    fig.tight_layout()
    return fig, axes

In [ ]:
# Full example view. For long examples this is dense, but useful for seeing global drift.
plot_phase(example_index=0, start=0, end=None);

In [ ]:
# Zoom to the first 6-state error. Set collapsed=True to zoom to the first downstream 4-track error.
start, end = first_error_window(example_index=0, pad=120, collapsed=False)
print(start, end)
plot_phase(example_index=0, start=start, end=end);

In [ ]:
# Inspect the latent tensor if you want to feed/compare it downstream.
print("encoded", tuple(encoded.shape))
print("logits", tuple(logits.shape))
print("probs", tuple(probs.shape))